# Volume 13, Chapter 12: Roth's theorem (Python companion)

Exercises first, solutions at the end. We compute $r_3(N)$ exactly for small $N$, build Behrend sets and find where they beat the ternary construction, look at the Fourier coefficients of progression-free sets (Lemma 12.2), and run one step of the density increment (Lemma 12.3). The computer science application: **Ruzsa–Szemerédi graphs**, built from progression-free sets, which show that testing triangle-freeness of a graph needs a number of samples that is not polynomial in $\frac1\varepsilon$. Requires `numpy`.

In [ ]:
import math, random, itertools
from collections import Counter
import numpy as np
random.seed(12)

def has_3ap(A):
    S = set(A); L = sorted(S)
    for i, x in enumerate(L):
        for z in L[i+1:]:
            if (x + z) % 2 == 0 and (x + z) // 2 in S: return True
    return False

def ternary01(N):
    # the m < N whose ternary digits are 0 or 1, shifted into [1, N]
    out = []
    for m in range(N):
        k = m
        while k and k % 3 != 2: k //= 3
        if k == 0: out.append(m + 1)
    return out

## Exercises

**P12.1 (★) Exact values.** Compute $r_3(N)$ for $N\le40$ by a search that only looks for sets containing both $1$ and $N$ (why is that enough?), and compare with the ternary construction.

**P12.2 (★★) Behrend against base $3$.** For $N=10^k$, maximize over $n$ and $d$ with $(2d)^n\le N$ the size $\max_k|S_k|$ of the most populated sphere (count lattice points on spheres by dynamic programming, not by enumeration). Compare the density with the ternary set's $\approx N^{\log2/\log3-1}$ and with Theorem 12.5's $e^{-4\sqrt{\log N}}$. Where does Behrend win?

**P12.3 (★★) Fourier coefficients.** For a progression-free set $A\subseteq[N]$ (ternary set, $N=3^7$) and a random set of the same density, compute $\max_{\xi\ne0}|\hat f(\xi)|$ in $\mathbb Z/M\mathbb Z$, with $f=\mathbf1_A-\alpha\mathbf1_{[N]}$ and $M$ an odd prime in $(2N,4N]$, and compare with Lemma 12.2's guarantee $\frac{\alpha^2N}{28}$. Count the $3$-APs of the random set and compare with Exercise 12.4.

**P12.4 (★★) One density increment.** Run the proof of Lemma 12.3 on the ternary set with $N=3^9$: take the largest Fourier coefficient, find $q$ by Dirichlet's principle, cut $[N]$ into progressions of difference $q$, and report the block where $A$ is densest. (The lemma's hypothesis $\alpha^2N\ge1792$ fails: we run the procedure anyway.)

**P12.5 (★★, computer science) Ruzsa–Szemerédi graphs.** From $A\subseteq[N]$ without $3$-AP build the tripartite graph with parts $X=[N]$, $Y=[2N]$, $Z=[3N]$ and, for each $x\in X$ and $a\in A$, the triangle $x\sim x+a\sim x+2a\sim x$ (edges $X$–$Y$ of difference $a$, $Y$–$Z$ of difference $a$, $X$–$Z$ of difference $2a$). Check that the graph has exactly $N|A|$ triangles, pairwise edge-disjoint. Deduce that it is far from triangle-free but has few triangles, and explain what this says about testing triangle-freeness by sampling.

## Solutions

### P12.1

In [ ]:
def r3_table(Nmax):
    r = [0, 1]
    for N in range(2, Nmax + 1):
        target = r[N - 1] + 1
        cnt = [0] * (N + 2); chosen = []
        def place(x, s):
            for y in chosen:
                for z in (2 * x - y, 2 * y - x):
                    if 1 <= z <= N: cnt[z] += s
                if (x + y) % 2 == 0: cnt[(x + y) // 2] += s
        def dfs(lo, size):
            if size == target: return True
            for y in range(lo, N):
                if r[N - y] < target - size: return False     # at most r_3(N-y) more elements fit in [y, N-1]
                if cnt[y]: continue
                place(y, 1); chosen.append(y)
                ok = dfs(y + 1, size + 1)
                chosen.pop(); place(y, -1)
                if ok: return True
            return False
        chosen.append(1); place(N, 1); chosen.append(N)
        r.append(target if (target <= 2 or dfs(2, 2)) else r[N - 1])
    return r
r = r3_table(40)
print("r_3(N), N=1..40:", r[1:])
print("ternary set:    ", [len(ternary01(N)) for N in range(1, 41)])

It is enough to look for sets containing $1$ and $N$: a progression-free subset of $[N]$ that misses $N$ lies in $[N-1]$, so it has at most $r_3(N-1)$ elements, and one that misses $1$ is a translate of a subset of $[N-1]$. So $r_3(N)=r_3(N-1)+1$ exactly when some progression-free set of that size contains both $1$ and $N$, and otherwise $r_3(N)=r_3(N-1)$. The pruning uses the table itself: elements still to be chosen in $[y,N-1]$ number at most $r_3(N-y)$.

The values agree with Figure 12.1 (computed the same way up to $70$) and with a brute-force check of all subsets for $N\le22$, made when this notebook was written. The ternary set is optimal for $N\le8$, for $10\le N\le19$ and for $38\le N\le40$, but loses up to $3$ elements in between ($r_3(27)=11$ against $8$): the greedy construction is far from optimal, even though nobody knows a formula for $r_3(N)$.

### P12.2

In [ ]:
from scipy.signal import fftconvolve
def sphere_counts(n, d):
    # approximate number of x in {0..d-1}^n with sum x_i^2 = k, for all k (polynomial powers by FFT)
    one = np.zeros((d - 1)**2 + 1)
    for t in range(d): one[t * t] += 1
    c = np.array([1.0])
    for _ in range(n): c = np.maximum(fftconvolve(c, one), 0)
    return c
def behrend_best(N, budget=3 * 10**6):
    best = (0, None)
    for n in range(2, 60):
        d = int(N ** (1 / n) / 2) + 1
        while d >= 1 and (2 * d)**n > N: d -= 1
        if d < 2: break
        if (d - 1)**2 * n > budget: continue
        size = float(max(sphere_counts(n, d)))
        if size > best[0]: best = (size, (n, d))
    return best
for k in [4, 6, 8, 12, 16, 24, 32, 48]:
    N = 10**k
    size, (n, d) = behrend_best(N)
    tern = 2**math.floor(math.log(N, 3))
    print(f"N=1e{k}: Behrend n={n:2d}, d={d:6d}: density {size/N:.2e};  ternary {tern/N:.2e};  e^(-4 sqrt(log N)) = {math.exp(-4*math.sqrt(math.log(N))):.2e}")

The ternary set wins for $N\le10^{12}$: its density decays only like $N^{-0.369}$, and Behrend's spheres need many dimensions before they pay off. Between $10^{12}$ and $10^{16}$ the curves cross ($9.2\cdot10^{-7}$ against $8.6\cdot10^{-7}$ at $10^{16}$), and from then on Behrend wins by a growing margin: at $10^{48}$ its density is $2\cdot10^{-11}$ against $10^{-18}$. The optimal dimension grows slowly, $n=17$ at $10^{48}$, roughly like $\sqrt{\log N}$ as in the proof ($\sqrt{\log10^{48}}\approx10.5$; the true optimum prefers larger $n$ and smaller $d$ than our simple choice). Theorem 12.5's bound $e^{-4\sqrt{\log N}}$ is far below the actual sizes: its constant $4$ is generous, and the best known constant is about $2\sqrt{2\log2}\approx2.35$, slightly improved in 2024.

The lesson matches Figure 12.2 of Chapter 11: asymptotic statements about additive structure often take effect only at sizes beyond any computer.

### P12.3

In [ ]:
def next_odd_prime(m):
    while True:
        m += 1
        if m % 2 and all(m % p for p in range(3, int(m**0.5) + 1, 2)): return m
N = 3**7
M = next_odd_prime(2 * N)
def maxcoef(A):
    alpha = len(A) / N
    f = np.zeros(M); f[np.array(A) - 1] = 1; f[:N] -= alpha     # x in [N] sits at index x-1
    F = np.fft.fft(f)
    k = 1 + np.argmax(np.abs(F[1:]))
    return alpha, np.abs(F[k]), k
A = ternary01(N)
R = sorted(random.sample(range(1, N + 1), len(A)))
for name, S in [("ternary set", A), ("random set", R)]:
    alpha, c, k = maxcoef(S)
    print(f"{name}: |A|={len(S)}, alpha={alpha:.4f}, max |f^(xi)| = {c:.1f} at xi={k} (M={M}); Lemma 12.2 guarantee alpha^2 N/28 = {alpha**2*N/28:.2f}")
# 3-APs with d != 0 in the random set, inside [N], against alpha^3 N^2 / 2
Rs = set(R)
aps = sum(1 for x in R for z in R if x < z and (x + z) % 2 == 0 and (x + z) // 2 in Rs)
print(f"random set: {aps} progressions with d>0, alpha^3 N^2/4 = {(len(R)/N)**3 * N**2 / 4:.1f}")

Lemma 12.2 guarantees a non-trivial coefficient of size at least $\frac{\alpha^2N}{28}=0.27$, a very weak bound here because $\alpha^2N=7.5$ is small. The progression-free ternary set has a coefficient of $77$, at $\xi=M-1$, the lowest frequency: the ternary set is biased towards the beginning of $[N]$ (half of it lies below $\frac N3$), and a bias at a low frequency is a bias on an interval. The random set has largest coefficient $29.6$, of the order $\sqrt{|A|\log M}$ expected for random signs, and $234$ progressions with $d>0$, close to the prediction $\frac{\alpha^3N^2}4=240$ (a quarter of $\alpha^3N^2$, because $x$ and $x+2d$ must both lie in $[N]$ and $d>0$).

### P12.4

In [ ]:
N = 3**9
M = next_odd_prime(2 * N)
A = ternary01(N); alpha = len(A) / N
f = np.zeros(M); f[np.array(A) - 1] = 1; f[:N] -= alpha
F = np.fft.fft(f)
xi = 1 + int(np.argmax(np.abs(F[1:]))); eta = abs(F[xi]) / N
print(f"N={N}, |A|={len(A)}, alpha={alpha:.4f}; largest coefficient at xi={xi}: |f^| = {eta:.4f} N")
for Q in [9, 81]:
    q = min(range(1, Q + 1), key=lambda q: abs(q * xi / M - round(q * xi / M)))
    L = max(1, N // (2 * q * 4))
    best = (0, None)
    # density of A on each block of L consecutive terms of each class
    Aset = np.zeros(N + 1, dtype=bool); Aset[A] = True
    for r0 in range(1, q + 1):
        cls = np.arange(r0, N + 1, q)
        for i in range(0, len(cls) - L + 1, L):
            dens = Aset[cls[i:i + L]].mean()
            if dens > best[0]: best = (dens, (r0, q, L, int(cls[i])))
    dens, (r0, q, L, start) = best
    print(f"Q={Q:3d}: q={q:3d}, ||q xi/M||={abs(q*xi/M - round(q*xi/M)):.4f}; densest block {start}, {start}+{q}, ... ({L} terms): density {dens:.4f} = {dens/alpha:.1f} alpha")

The largest coefficient is at $\xi=M-1$, so $\frac{\xi}M\approx-\frac1M$ is already close to an integer: Dirichlet's principle returns $q=1$ for every $Q$, the progressions are intervals, and the densest block is the initial interval of $2460$ terms, where the ternary set has density $0.069=2.7\alpha$. This is the increment of Lemma 12.3 in its simplest form: the set was not spread evenly, and restricting to the right interval raises its density. (For the ternary set, restricting to $[1,3^k]$ for smaller $k$ raises the density further, by a factor $\frac32$ at each step: the set is self-similar, and the iteration of Theorem 12.4 would run down this tower.)

### P12.5

In [ ]:
def rs_graph(N, A):
    E = set()
    for x in range(1, N + 1):
        for a in A:
            E.add((("X", x), ("Y", x + a))); E.add((("Y", x + a), ("Z", x + 2 * a))); E.add((("X", x), ("Z", x + 2 * a)))
    return E
def count_triangles(N, A, E):
    adjXY = Counter(); adj = {}
    for u, v in E: adj.setdefault(u, set()).add(v)
    t = 0
    for x in range(1, N + 1):
        for y in adj.get(("X", x), ()):
            if y[0] != "Y": continue
            for z in adj.get(y, ()):
                if z in adj[("X", x)]: t += 1
    return t
for k in [3, 4, 5]:
    N = 3**k; A = [a for a in ternary01(N)]
    E = rs_graph(N, A); T = count_triangles(N, A, E)
    n = N + 2 * N + 3 * N
    print(f"N={N}: |A|={len(A)}, vertices {n}, edges {len(E)} = 3N|A|: {len(E) == 3*N*len(A)}, triangles {T} = N|A|: {T == N*len(A)}; "
          f"edges to delete >= {N*len(A)} = {N*len(A)/n**2:.4f} n^2, triangle density {T/math.comb(n, 3):.2e}")

The edges come in triangles $\{x,\,x+a,\,x+2a\}$, one for each $x\in X$ and $a\in A$; the three edges of one triangle determine $x$ and $a$, so these $N|A|$ triangles are edge-disjoint and there are $3N|A|$ edges. Any triangle $x\sim y\sim z$ of the graph has $y=x+a$, $z=y+b$ and $z=x+2c$ with $a,b,c\in A$, so $a+b=2c$: since $A$ has no $3$-AP, $a=b=c$, and the triangle is one of the planted ones. The output confirms both counts.

To make the graph triangle-free one must delete at least one edge from each of the $N|A|$ disjoint triangles: with $n=6N$ vertices, $N|A|=\frac{\alpha}{36}n^2$ edges, so the graph is $\varepsilon$-far from triangle-free with $\varepsilon\approx\frac\alpha{36}$. Yet its triangle density is tiny: $\frac{N|A|}{\binom n3}\approx\frac{\alpha}{36n}$. A *property tester* samples a few vertices and accepts if it sees no triangle; on this graph it needs about $\frac1{\text{triangle density}}$ samples. With Behrend's sets, $\alpha=e^{-c\sqrt{\log N}}$, so $\varepsilon$ is only quasi-polynomially small in $n$ while the triangle density is $\frac1n$ times smaller still, and Noga Alon deduced in 2002 that every tester for triangle-freeness needs a number of queries that grows faster than any polynomial in $\frac1\varepsilon$, roughly $(\frac1\varepsilon)^{c\log\frac1\varepsilon}$. The same graphs, which are unions of $N$ induced matchings, give lower bounds in streaming algorithms and communication complexity.